# Kaggle Training: LogFilter Classifier

This notebook trains the XGBoost log classifier using the existing project scripts. The notebook is for Kaggle execution and artifact export; reusable logic should stay in `training/`.


## 1. Locate the repo

Run this notebook from the project root when possible. If Kaggle places the repo inside `/kaggle/working`, the cell below tries to find it.


In [2]:
from pathlib import Path
import json
import os
import subprocess
import sys

KAGGLE_WORKING = Path('/kaggle/working')
REPO_DIR = Path.cwd()

if not (REPO_DIR / 'training' / 'train.py').exists():
    matches = list(KAGGLE_WORKING.glob('*/training/train.py'))
    if matches:
        REPO_DIR = matches[0].parents[1]

if not (REPO_DIR / 'training' / 'train.py').exists():
    raise FileNotFoundError('Could not find training/train.py. Upload or clone the repo into /kaggle/working first.')

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / 'src'))
print('Repo:', REPO_DIR)


Repo: /home/jacob/Projects/Modern-AI-Log-Filter-Training


## 2. Install training dependencies

Kaggle usually has several packages preinstalled, but this cell makes the training/export dependencies explicit.


In [ ]:
%pip install -q numpy pandas scikit-learn xgboost onnx onnxruntime onnxmltools skl2onnx


## 3. Attach the dataset

The training script expects `HDFS_v3_TraceBench/preprocessed/normal_trace.csv` and `failure_trace.csv` under the repo. If the files are attached as a Kaggle dataset, this cell symlinks the Kaggle input path into the expected project location.


In [ ]:
PREPROCESSED_DIR = REPO_DIR / 'HDFS_v3_TraceBench' / 'preprocessed'

def has_trace_files(path: Path) -> bool:
    return (path / 'normal_trace.csv').exists() and (path / 'failure_trace.csv').exists()

if not has_trace_files(PREPROCESSED_DIR):
    input_root = Path('/kaggle/input')
    candidates = [p for p in input_root.rglob('preprocessed') if has_trace_files(p)]
    if not candidates:
        raise FileNotFoundError('No Kaggle input folder with normal_trace.csv and failure_trace.csv was found.')

    source = candidates[0]
    PREPROCESSED_DIR.parent.mkdir(parents=True, exist_ok=True)
    if PREPROCESSED_DIR.exists() and not PREPROCESSED_DIR.is_symlink():
        raise FileExistsError(f'{PREPROCESSED_DIR} exists but does not contain the expected trace files.')
    if not PREPROCESSED_DIR.exists():
        os.symlink(source, PREPROCESSED_DIR, target_is_directory=True)
    print('Linked dataset:', source, '->', PREPROCESSED_DIR)
else:
    print('Dataset already available:', PREPROCESSED_DIR)


## 4. Sampled training run

Start with a smaller run to verify the environment and artifact export.


In [ ]:
sample_cmd = [
    sys.executable,
    'training/train.py',
    '--sample-normal', '50000',
    '--sample-failure', '10000',
    '--output-dir', 'models',
]
print(' '.join(sample_cmd))
subprocess.run(sample_cmd, check=True)


## 5. Full training run

Run this only after the sampled training succeeds. Uncomment the cell body when ready.


In [ ]:
# full_cmd = [
#     sys.executable,
#     'training/train.py',
#     '--output-dir', 'models',
# ]
# print(' '.join(full_cmd))
# subprocess.run(full_cmd, check=True)


## 6. Inspect artifacts


In [ ]:
models_dir = REPO_DIR / 'models'
for path in sorted(models_dir.glob('*')):
    if path.is_file():
        print(f'{path.name}: {path.stat().st_size:,} bytes')

metrics_path = models_dir / 'training_metrics.json'
if metrics_path.exists():
    print(json.dumps(json.loads(metrics_path.read_text()), indent=2))


## 7. Package model artifacts

The zip is written to `/kaggle/working/` so it appears in Kaggle output artifacts.


In [ ]:
import zipfile

artifact_names = [
    'log_classifier.onnx',
    'log_classifier.json',
    'scaler.json',
    'feature_names.json',
    'training_metrics.json',
]
zip_path = KAGGLE_WORKING / 'logfilter-model-artifacts.zip'

with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for name in artifact_names:
        path = models_dir / name
        if path.exists():
            zf.write(path, arcname=f'models/{name}')

print('Artifact zip:', zip_path)
